# NEUROMOYO — Sahara CodeSwitch Africa Challenge
## Clean benchmark notebook — start from scratch

This notebook is intentionally lightweight. It downloads the AfriSwitch Pidgin test split, selects a reproducible evaluation subset, runs the required speech-model comparison, and produces the evidence needed for the hackathon submission.

**Models:** Sahara + Whisper Tiny + Whisper Base + MMS 1B.

**Important:** ASR benchmark performance is not clinical validation. Clinical Information Preservation is a proposed NEUROMOYO metric, not an official Intron metric.

## 1. Before running

Add these to **Google Colab Secrets**:
- `HF_TOKEN` — Hugging Face token with access to AfriSwitch.
- `SAHARA_API_KEY` — Sahara API key.

Do not put either secret in the notebook source or Git repository.

In [ ]:
!pip -q install -U datasets huggingface_hub transformers accelerate jiwer librosa soundfile pandas numpy scipy matplotlib requests tqdm


In [ ]:
import os, re, json, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/content/neuromoyo_sahara_challenge')
for p in ['data','audio','results','figures','report']:
    (ROOT/p).mkdir(parents=True, exist_ok=True)

N_EVAL = 50                 # change to 100/200 when resources allow
SEED = 20260909
RUN_SAHARA = True
RUN_WHISPER = True
RUN_MMS = True

DATASET = 'intronhealth/AfriSwitch'
CONFIG = 'pidgin'
SPLIT = 'test'

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    SAHARA_API_KEY = userdata.get('SAHARA_API_KEY')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    SAHARA_API_KEY = os.environ.get('SAHARA_API_KEY')

assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets.'
if RUN_SAHARA: assert SAHARA_API_KEY, 'Add SAHARA_API_KEY to Colab Secrets.'
print('Configuration ready:', ROOT)

## 2. Load and inspect AfriSwitch

AfriSwitch is used as the public, reproducible code-switching benchmark. We will explicitly show judges:
1. what dataset was used,
2. which split was used,
3. how many utterances were available,
4. how the evaluation subset was selected,
5. what every model received.

In [ ]:
from datasets import load_dataset, Audio

ds = load_dataset(DATASET, CONFIG, split=SPLIT, token=HF_TOKEN)
meta = ds.remove_columns(['audio']).to_pandas()
meta['sample_index'] = np.arange(len(meta))
meta['is_code_switched'] = meta['num_switch_points'] > 0

print(ds)
print('Total utterances:', len(ds))
display(meta[['language','duration','cmi','num_switch_points','is_code_switched']].describe(include='all'))

In [ ]:
# Reproducible stratified sample.
# We balance code-switch status and spread the sample across CMI, duration and switch counts.

work = meta.copy()
work['cmi_bin'] = pd.qcut(work['cmi'], 4, labels=False, duplicates='drop')
work['duration_bin'] = pd.qcut(work['duration'], 4, labels=False, duplicates='drop')
work['switch_bin'] = pd.cut(work['num_switch_points'], [-1,0,1,2,np.inf], labels=['0','1','2','3+'])
work['stratum'] = work[['is_code_switched','cmi_bin','duration_bin','switch_bin']].astype(str).agg('|'.join, axis=1)

rng = np.random.default_rng(SEED)
selected=[]
groups = list(work.groupby('stratum'))
base = N_EVAL // len(groups)
remainder = N_EVAL % len(groups)
for i,(g,x) in enumerate(groups):
    k = base + (1 if i < remainder else 0)
    k = min(k, len(x))
    if k:
        selected.append(x.sample(k, random_state=int(rng.integers(0,2**31-1))))

eval_meta = pd.concat(selected).drop_duplicates('sample_index')
if len(eval_meta) < N_EVAL:
    remaining = work[~work.sample_index.isin(eval_meta.sample_index)]
    eval_meta = pd.concat([eval_meta, remaining.sample(N_EVAL-len(eval_meta), random_state=SEED)])
eval_meta = eval_meta.sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(eval_meta) == N_EVAL

eval_meta.to_csv(ROOT/'data/evaluation_manifest.csv', index=False)
display(eval_meta[['sample_index','duration','cmi','num_switch_points','is_code_switched']].head(10))
display(eval_meta['is_code_switched'].value_counts().rename({False:'non-code-switched',True:'code-switched'}))

In [ ]:
# Materialize only the selected audio files.
audio_ds = ds.cast_column('audio', Audio(sampling_rate=16000))
import soundfile as sf

rows=[]
for _, r in eval_meta.iterrows():
    idx=int(r.sample_index)
    a=audio_ds[idx]['audio']
    out=ROOT/'audio'/f'{idx:05d}.wav'
    sf.write(out, a['array'], int(a['sampling_rate']))
    rows.append({'sample_index':idx,'audio_path':str(out),'reference':str(r.transcription),
                 'duration_s':float(r.duration),'cmi':float(r.cmi),
                 'num_switch_points':int(r.num_switch_points),
                 'is_code_switched':bool(r.is_code_switched)})
manifest=pd.DataFrame(rows)
manifest.to_csv(ROOT/'data/audio_manifest.csv', index=False)
print('Audio files:', len(manifest))

## 3. Sahara

The benchmark uses the official Sahara file-upload flow. We record the complete latency and transcript for each utterance. If an API call fails, the row is not silently discarded; the error is recorded so the final report can show success/failure counts.

In [ ]:
import requests, time

UPLOAD='https://infer.voice.intron.io/file/v1/upload/sync'
STATUS='https://infer.voice.intron.io/file/v1/status/{}'
HEADERS={'Authorization':f'Bearer {SAHARA_API_KEY}'}

def sahara_transcribe(path):
    with open(path,'rb') as f:
        t0=time.perf_counter()
        r=requests.post(UPLOAD, headers=HEADERS,
            data={'audio_file_name':Path(path).name},
            files={'audio_file_blob':(Path(path).name,f,'audio/wav')}, timeout=180)
        elapsed=time.perf_counter()-t0
    payload=r.json()
    if r.status_code == 200:
        d=payload.get('data',{})
        return d.get('audio_transcript',''), elapsed, d.get('file_id'), None
    fid=payload.get('data',{}).get('file_id')
    if r.status_code == 503 and fid:
        while True:
            s=requests.get(STATUS.format(fid),headers=HEADERS,timeout=60).json().get('data',{})
            if s.get('processing_status')=='FILE_TRANSCRIBED':
                return s.get('audio_transcript',''), time.perf_counter()-t0, fid, None
            if s.get('processing_status')=='FILE_PROCESSING_FAILED':
                return '', time.perf_counter()-t0, fid, str(s)
            if time.perf_counter()-t0>240: return '',time.perf_counter()-t0,fid,'timeout'
            time.sleep(3)
    return '', elapsed, fid, str(payload)

def run_sahara():
    out=ROOT/'results/sahara.jsonl'
    records=[]
    for _,r in manifest.iterrows():
        text,lat,fid,error=sahara_transcribe(r.audio_path)
        records.append({**r.to_dict(),'model':'Sahara','hypothesis':text,'latency_s':lat,'file_id':fid,'error':error})
        pd.DataFrame(records).to_json(out,orient='records',lines=True)
    return pd.DataFrame(records)

if RUN_SAHARA:
    sahara_results=run_sahara()
    print('Sahara complete:', len(sahara_results), 'rows')

## 4. Whisper Tiny + Base


In [ ]:
import torch
from transformers import pipeline

device=0 if torch.cuda.is_available() else -1
dtype=torch.float16 if device==0 else torch.float32

def run_whisper(model_id, filename):
    pipe=pipeline('automatic-speech-recognition', model=model_id, device=device, torch_dtype=dtype)
    records=[]
    for _,r in manifest.iterrows():
        t0=time.perf_counter()
        z=pipe(r.audio_path, generate_kwargs={'task':'transcribe'})
        lat=time.perf_counter()-t0
        records.append({**r.to_dict(),'model':model_id.split('/')[-1], 'hypothesis':z['text'], 'latency_s':lat, 'error':None})
        pd.DataFrame(records).to_json(ROOT/f'results/{filename}',orient='records',lines=True)
    return pd.DataFrame(records)

if RUN_WHISPER:
    whisper_tiny=run_whisper('openai/whisper-tiny','whisper_tiny.jsonl')
    whisper_base=run_whisper('openai/whisper-base','whisper_base.jsonl')
    print('Whisper complete')

## 5. MMS 1B

MMS is included as an independent multilingual baseline. For this Pidgin experiment, document the language-adapter limitation clearly rather than presenting it as a dedicated Cameroon Pidgin model.


In [ ]:
from transformers import AutoProcessor, AutoModelForCTC
import librosa

def run_mms():
    processor=AutoProcessor.from_pretrained('facebook/mms-1b-all')
    model=AutoModelForCTC.from_pretrained('facebook/mms-1b-all')
    dev='cuda' if torch.cuda.is_available() else 'cpu'
    model.to(dev).eval()
    records=[]
    for _,r in manifest.iterrows():
        speech,_=librosa.load(r.audio_path,sr=16000,mono=True)
        inputs=processor(speech,sampling_rate=16000,return_tensors='pt')
        inputs={k:v.to(dev) for k,v in inputs.items()}
        t0=time.perf_counter()
        with torch.no_grad(): logits=model(**inputs).logits
        ids=torch.argmax(logits,dim=-1)
        text=processor.batch_decode(ids)[0]
        lat=time.perf_counter()-t0
        records.append({**r.to_dict(),'model':'MMS-1B','hypothesis':text,'latency_s':lat,
                        'error':None,'model_note':'Available multilingual pathway; not a dedicated Cameroon Pidgin adapter.'})
        pd.DataFrame(records).to_json(ROOT/'results/mms.jsonl',orient='records',lines=True)
    return pd.DataFrame(records)

if RUN_MMS:
    mms_results=run_mms()
    print('MMS complete')

## 6. Score every model on the exact same utterances


In [ ]:
from jiwer import wer, cer
from scipy.stats import spearmanr

def norm(x):
    x=str(x).lower().strip()
    x=re.sub(r"[^a-z0-9' ]+",' ',x)
    return re.sub(r'\s+',' ',x).strip()

frames=[]
for fn in ['sahara.jsonl','whisper_tiny.jsonl','whisper_base.jsonl','mms.jsonl']:
    p=ROOT/'results'/fn
    if p.exists(): frames.append(pd.read_json(p,lines=True))
results=pd.concat(frames,ignore_index=True)
results['wer']=results.apply(lambda x:wer(norm(x.reference),norm(x.hypothesis)) if not x.get('error') else np.nan,axis=1)
results['cer']=results.apply(lambda x:cer(norm(x.reference),norm(x.hypothesis)) if not x.get('error') else np.nan,axis=1)
results['rtf']=results['latency_s']/results['duration_s']
results.to_csv(ROOT/'results/all_results.csv',index=False)

summary=results.groupby('model').agg(n=('sample_index','count'),successful=('wer','count'),
    mean_WER=('wer','mean'),median_WER=('wer','median'),mean_CER=('cer','mean'),
    mean_latency_s=('latency_s','mean'),mean_RTF=('rtf','mean')).sort_values('mean_WER')
display(summary)

In [ ]:
rows=[]
for model,d in results.groupby('model'):
    cs=d[d.is_code_switched]
    non=d[~d.is_code_switched]
    rows.append({'model':model,'CS_n':len(cs),'non_CS_n':len(non),
                 'CS_WER':cs.wer.mean(),'non_CS_WER':non.wer.mean(),
                 'CS_penalty':cs.wer.mean()-non.wer.mean(),
                 'CMI_WER_rho':spearmanr(d.cmi,d.wer,nan_policy='omit').statistic,
                 'switch_WER_rho':spearmanr(d.num_switch_points,d.wer,nan_policy='omit').statistic})
cs_table=pd.DataFrame(rows)
display(cs_table)

## 7. Lightweight uncertainty analysis

We use bootstrap confidence intervals for mean WER. Because every model sees the same utterances, we also calculate paired Sahara-vs-baseline differences.

In [ ]:
def bootstrap_mean(x, B=2000):
    x=np.asarray(x.dropna(),float); rng=np.random.default_rng(SEED)
    samples=rng.choice(x,(B,len(x)),replace=True).mean(axis=1)
    return x.mean(),np.quantile(samples,.025),np.quantile(samples,.975)

ci=[]
for model,d in results.groupby('model'):
    m,lo,hi=bootstrap_mean(d.wer)
    ci.append({'model':model,'mean_WER':m,'CI_low':lo,'CI_high':hi})
display(pd.DataFrame(ci).sort_values('mean_WER'))

## 8. Clinical Information Preservation — annotation layer

Do **not** infer clinical ground truth from ordinary AfriSwitch utterances. For the health story, annotate a small, consented local set or use the simulated AfriSwitchCare dataset.

Recommended fields: symptom, body site, duration, frequency, severity, negation, medication, functional change.

The final submission should show that the selected ASR is used for a downstream task — for example, converting a patient's natural code-switched description into a structured referral summary — rather than stopping at transcription.

In [ ]:
clinical_fields=['symptom','body_site','duration','frequency','severity','negation','medication','functional_change']
clinical_template=pd.DataFrame(columns=['sample_id','reference_text','hypothesis_text']+
    [f'ref_{x}' for x in clinical_fields]+[f'hyp_{x}' for x in clinical_fields])
clinical_template.to_csv(ROOT/'data/clinical_annotations_template.csv',index=False)
print(ROOT/'data/clinical_annotations_template.csv')

## 9. Generate figures and submission-ready evidence


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9,5))
plt.bar(summary.index,summary.mean_WER)
plt.ylabel('Mean WER'); plt.title('NEUROMOYO — Speech Model Benchmark'); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(ROOT/'figures/mean_wer.png',dpi=200); plt.show()

plt.figure(figsize=(9,5))
plot=results.groupby(['model','is_code_switched']).wer.mean().unstack()
plot.plot(kind='bar',ax=plt.gca())
plt.ylabel('Mean WER'); plt.title('Code-Switching Robustness'); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(ROOT/'figures/code_switch_penalty.png',dpi=200); plt.show()

In [ ]:
report = f'''# NEUROMOYO — Sahara CodeSwitch Africa Benchmark

## Dataset
- Dataset: {DATASET}
- Configuration: {CONFIG}
- Split: {SPLIT}
- Full split size: {len(meta):,} utterances
- Evaluation size: {len(eval_meta):,} utterances
- Sampling seed: {SEED}

## Models
- Intron Sahara
- Whisper Tiny
- Whisper Base
- Meta MMS 1B

## Overall results

{summary.to_markdown()}

## Code-switch analysis

{cs_table.to_markdown(index=False)}

## Interpretation
The benchmark compares all models on the same selected utterances. WER/CER measure transcription quality; latency and RTF measure efficiency. Code-switch penalty measures the difference between code-switched and non-code-switched subsets. Correlations with CMI and switch count are descriptive and should not be interpreted causally.

## Clinical downstream task
Clinical Information Preservation is a proposed NEUROMOYO metric. It must be evaluated only on appropriately annotated clinical-domain data, such as simulated AfriSwitchCare conversations or consented local recordings.

## Limitations
This Pidgin evaluation is not representative of all Cameroon or Africa. MMS is not being presented as a dedicated Cameroon Pidgin system. ASR benchmark results are not clinical validation and do not establish diagnostic accuracy.
'''
(ROOT/'report/BENCHMARK_REPORT.md').write_text(report)
print(report)

# 10. What judges should see in the final submission

### Data → Use → Evidence
**AfriSwitch:** public code-switching benchmark → used to compare speech models fairly.

**AfriSwitchCare / consented local clinical speech:** clinical downstream evaluation → used to test whether transcripts preserve medically relevant information.

**Existing NEUROMOYO Parkinsonian speech model:** neurological signal analysis → used after ASR/audio capture for the neurological screening workflow.

**Sahara + baselines:** model selection → benchmark evidence determines which ASR pathway is used in the product.

That creates a clean story: **data → model comparison → model selection → downstream clinical task → neurological workflow → human referral/decision support.**

Do not claim that the benchmark dataset itself diagnoses Parkinson's disease.

# 11. Final checklist

- [ ] Same utterances evaluated by all models
- [ ] Sahara + at least 2 other models (we use 3 baselines)
- [ ] Dataset, split and sampling method shown
- [ ] WER and CER reported
- [ ] Code-switch penalty reported
- [ ] Latency/RTF reported
- [ ] Uncertainty/error analysis included
- [ ] Clinical downstream evaluation separated from ASR benchmark
- [ ] No raw health audio committed to Git
- [ ] No API key committed to Git
- [ ] Consent/de-identification documented for any local recordings
- [ ] Responsible AI note included
- [ ] Working prototype demonstrates a downstream task, not transcription alone
